# EDA - NYC Taxi Fare Prediction (May 2022)
## Cami's Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

In [ ]:
# Load the data
data_path = Path("../data/raw/yellow_tripdata_2022-05.parquet")
df = pd.read_parquet(data_path)

print(f"Data loaded successfully")
print(f"Shape: {df.shape}")

In [ ]:
# Basic information about the dataset
print("=== Dataset Overview ===")
print(f"Number of samples: {df.shape[0]:,}")
print(f"Number of features: {df.shape[1]}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display first few rows
print("=== First 5 rows ===")
df.head()

In [ ]:
# Column information
print("=== Column Information ===")
df.info()

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Statistical summary
print("=== Statistical Summary ===")
df.describe()

In [ ]:
# Check data types
print("=== Data Types ===")
print(df.dtypes)

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates:,}")
print(f"Percentage of duplicates: {(duplicates/len(df))*100:.2f}%")

## Target Variables Analysis
Focus on fare_amount and trip_duration (need to calculate duration)

In [ ]:
# Calculate trip duration
df['trip_duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60  # in minutes

print("=== Target Variables Statistics ===")
print("Fare Amount:")
print(df['fare_amount'].describe())
print("\nTrip Duration (minutes):")
print(df['trip_duration'].describe())

In [ ]:
# Distribution of fare amount
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Fare amount distribution
sns.histplot(df['fare_amount'], bins=50, ax=axes[0])
axes[0].set_title('Distribution of Fare Amount')
axes[0].set_xlabel('Fare Amount ($)')
axes[0].set_ylabel('Frequency')

# Trip duration distribution
sns.histplot(df['trip_duration'], bins=50, ax=axes[1])
axes[1].set_title('Distribution of Trip Duration (minutes)')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Feature Analysis
Analyze key features for prediction

In [ ]:
# Extract temporal features
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day'] = df['tpep_pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month

print("=== Temporal Features Extracted ===")
print(df[['pickup_hour', 'pickup_day', 'pickup_month']].head())

In [ ]:
# Analyze pickup hour distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='pickup_hour')
plt.title('Distribution of Pickup Hours')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
# Analyze pickup day distribution
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='pickup_day')
plt.title('Distribution of Pickup Days')
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.xticks(range(7), day_names)
plt.show()

In [ ]:
# Passenger count analysis
print("=== Passenger Count Statistics ===")
print(df['passenger_count'].describe())
print(f"\nUnique passenger counts: {sorted(df['passenger_count'].unique())}")

In [ ]:
# Passenger count distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='passenger_count')
plt.title('Distribution of Passenger Count')
plt.xlabel('Number of Passengers')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
# Distance analysis
print("=== Trip Distance Statistics ===")
print(df['trip_distance'].describe())

In [ ]:
# Trip distance distribution
plt.figure(figsize=(12, 6))
sns.histplot(df['trip_distance'], bins=50)
plt.title('Distribution of Trip Distance (miles)')
plt.xlabel('Distance (miles)')
plt.ylabel('Frequency')
plt.xlim(0, 20)  # Focus on reasonable distances
plt.show()

## Correlation Analysis
Analyze relationships between features

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['fare_amount', 'trip_distance', 'passenger_count', 
                'trip_duration', 'pickup_hour', 'pickup_day']

correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Outlier Detection
Identify potential outliers in key variables

In [ ]:
# Box plots for key variables
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Fare amount
sns.boxplot(data=df, y='fare_amount', ax=axes[0, 0])
axes[0, 0].set_title('Fare Amount Distribution')
axes[0, 0].set_ylabel('Fare ($)')

# Trip distance
sns.boxplot(data=df, y='trip_distance', ax=axes[0, 1])
axes[0, 1].set_title('Trip Distance Distribution')
axes[0, 1].set_ylabel('Distance (miles)')

# Trip duration
sns.boxplot(data=df, y='trip_duration', ax=axes[1, 0])
axes[1, 0].set_title('Trip Duration Distribution')
axes[1, 0].set_ylabel('Duration (minutes)')

# Passenger count
sns.boxplot(data=df, y='passenger_count', ax=axes[1, 1])
axes[1, 1].set_title('Passenger Count Distribution')
axes[1, 1].set_ylabel('Number of Passengers')

plt.tight_layout()
plt.show()

## Data Quality Issues Identified
Summary of potential issues found during EDA

In [ ]:
# Check for unrealistic values
print("=== Data Quality Checks ===")

# Negative fares
negative_fares = (df['fare_amount'] < 0).sum()
print(f"Negative fare amounts: {negative_fares:,}")

# Zero or negative distances
invalid_distances = (df['trip_distance'] <= 0).sum()
print(f"Zero or negative distances: {invalid_distances:,}")

# Negative durations
negative_durations = (df['trip_duration'] < 0).sum()
print(f"Negative trip durations: {negative_durations:,}")

# Very long durations (more than 4 hours)
long_durations = (df['trip_duration'] > 240).sum()
print(f"Very long durations (>4 hours): {long_durations:,}")

# Zero passenger count
zero_passengers = (df['passenger_count'] == 0).sum()
print(f"Zero passenger count: {zero_passengers:,}")

## Initial Recommendations for Data Cleaning
Based on EDA findings

### Cleaning Steps to Consider:
1. **Remove negative fare amounts** - These are likely data errors
2. **Filter invalid distances** - Remove trips with distance <= 0
3. **Handle unrealistic durations** - Filter negative or extremely long durations
4. **Address missing values** - Decide on imputation or removal strategy
5. **Handle outliers** - Consider IQR method or domain knowledge for fare/distance
6. **Zero passenger handling** - Decide whether to keep or remove these records

### Feature Engineering Ideas:
1. **Temporal features** - Hour, day of week, month (already extracted)
2. **Speed calculation** - distance/duration (handle division by zero)
3. **Location features** - Pickup/dropoff borough or zones (if location data available)
4. **Rush hour indicator** - Based on hour and day
5. **Weekend indicator** - Based on day of week

### Next Steps:
1. Implement data cleaning pipeline
2. Create feature engineering functions
3. Save cleaned dataset for model training
4. Compare results with team members on Friday